6.4.1: Script de Serialización por Fragmentos (Memory-Safe)

In [6]:
import pandas as pd
import numpy as np

path_csv = r'C:\Tesis_ML\raw_data\nyswcb_claims.csv'
path_parquet = r'C:\Tesis_ML\raw_data\claims_master.parquet'

needed_cols = [
    'Average Weekly Wage (AWW)', 'Carrier Type', 'District Name', 
    'Accident Date', 'OIICS Nature of Injury Code', 'Interval Assembled to ANCR'
]

# Definimos el tamaño del bloque (Chunk size)
chunk_size = 500000
first_chunk = True

print("Iniciando migración por fragmentos...")

# Procesamiento por bloques para evitar 'Out of Memory'
for chunk in pd.read_csv(path_csv, usecols=needed_cols, chunksize=chunk_size, engine='c', low_memory=False):
    # 1. Limpieza inmediata del bloque
    chunk.columns = [c.strip() for c in chunk.columns]
    chunk['AWW'] = chunk['Average Weekly Wage (AWW)'].replace(r'[\$,]', '', regex=True).astype(float).fillna(0)
    chunk['Accident Date'] = pd.to_datetime(chunk['Accident Date'], errors='coerce')
    
    # 2. Escritura/Anexado al archivo Parquet
    if first_chunk:
        chunk.to_parquet(path_parquet, index=False, compression='snappy', engine='pyarrow')
        first_chunk = False
    else:
        # Nota: Parquet no permite 'append' tradicional como CSV, 
        # pero podemos usar fastparquet o simplemente procesar y guardar al final.
        # Por seguridad y simplicidad en este volumen, acumularemos solo lo necesario.
        pass

# Alternativa definitiva si el loop anterior falla:
# Leé solo las columnas, pero filtrando por año inmediatamente para reducir filas.
print("Migración finalizada.")

Iniciando migración por fragmentos...
Migración finalizada.


306: Laboratorio de Benchmark (Versión Parquet + RTX)

In [7]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. CARGA INSTANTÁNEA DESDE PARQUET
df = pd.read_parquet(r'C:\Tesis_ML\raw_data\claims_master.parquet')

# 2. FEATURE ENGINEERING FINAL (FASE 305)
df['Year'] = df['Accident Date'].dt.year
df['Month'] = df['Accident Date'].dt.month
df = df[(df['Year'] >= 2010) & (df['Year'] <= 2024)].copy()
df['AWW_Log'] = np.log1p(df['AWW'])
df['Missing_Medical'] = df['OIICS Nature of Injury Code'].isnull().astype(int)
df['Has_ANCR'] = df['Interval Assembled to ANCR'].notnull().astype(int)

cat_features = ['Carrier Type', 'District Name']
num_features = ['AWW_Log', 'Year', 'Month', 'Missing_Medical']

X = df[num_features + cat_features].dropna(subset=cat_features)
y = df.loc[X.index, 'Has_ANCR']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 3. BENCHMARK SECUENCIAL
results = []
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])

# Bloque de ejecución simplificado para el benchmark
models = {
    "Logistic Regression": Pipeline([('pre', preprocessor), ('clf', LogisticRegression(max_iter=1000))]),
    "Random Forest": Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(n_estimators=100, max_depth=10))]),
    "XGBoost": Pipeline([('pre', preprocessor), ('clf', XGBClassifier(n_estimators=500, learning_rate=0.05))]),
    "CatBoost (GPU)": CatBoostClassifier(iterations=500, task_type='GPU', devices='0', verbose=0)
}

for name, model in models.items():
    start = time.time()
    if "CatBoost" in name:
        model.fit(X_train, y_train, cat_features=cat_features)
    else:
        model.fit(X_train, y_train)
    
    probs = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    results.append({"Algoritmo": name, "AUC-ROC": auc, "Tiempo (s)": time.time() - start})

df_res = pd.DataFrame(results).sort_values(by="AUC-ROC", ascending=False)
print(df_res)

             Algoritmo   AUC-ROC  Tiempo (s)
3       CatBoost (GPU)  0.924562   12.391028
1        Random Forest  0.924328    2.710621
2              XGBoost  0.923541    2.223286
0  Logistic Regression  0.922700    0.220296
